In [6]:
import gc
import json
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABEL_COLS = ['ACL', 'PCL', 'MCL', 'LCL', 'Medial_Meniscus', 'Lateral_Meniscus', 'Effusion']

SYSTEM_PROMPT = """You are a multilingual expert radiologist extracting findings from knee MRI reports in ANY language.

For each condition, output:
- 1: Explicitly positive / confirmed tear, injury, or effusion.
- 0: Explicitly normal, intact, absent, or negated (e.g., "sin rotura", "normal", "intacto", "unremarkable").
- null: Ambiguous, unclear, not mentioned, or uncertain.

CRITICAL: If uncertain or not explicitly stated, output null.

Respond ONLY with a JSON object containing keys and not a single extra word:
{"ACL": int_or_null, "PCL": int_or_null, "MCL": int_or_null, "LCL": int_or_null, "Medial_Meniscus": int_or_null, "Lateral_Meniscus": int_or_null, "Effusion": int_or_null}"""

# -------------------------------------------------------------------
# Helper: Generic Batched Inference Generator
# -------------------------------------------------------------------
def run_model_inference(model_name: str, reports: list, uids: list, batch_size: int = 16) -> pd.DataFrame:
    print(f"\n--- Loading Model: {model_name} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=False
    )
    model.eval()

    results = []

    for i in tqdm(range(0, len(reports), batch_size), desc=f"Inferencing {model_name.split('/')[-1]}"):
        batch_reports = reports[i:i+batch_size]
        batch_uids = uids[i:i+batch_size]

        # Universal message formatting: Combine system prompt into user message
        # Works on Gemma, Qwen, Llama, Mistral, and Phi without template errors
        prompts = [
            tokenizer.apply_chat_template(
                [{
                    "role": "user",
                    "content": f"{SYSTEM_PROMPT}\n\nMRI Report:\n\"\"\"{r}\"\"\""
                }],
                tokenize=False,
                add_generation_prompt=True
            ) for r in batch_reports
        ]

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=96,
                temperature=0.01,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        input_len = inputs['input_ids'].shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

        for uid, resp in zip(batch_uids, decoded):
            row = {'StudyInstanceUID': uid}
            try:
                match = re.search(r'\{.*?\}', resp, re.DOTALL)
                parsed = json.loads(match.group(0)) if match else json.loads(resp)
            except Exception:
                parsed = {}

            for col in LABEL_COLS:
                val = parsed.get(col, None)
                row[col] = int(val) if val in [0, 1] else None
            results.append(row)

    # Clean VRAM after model execution
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

    return pd.DataFrame(results).set_index('StudyInstanceUID')


# -------------------------------------------------------------------
# Execution: Sequential Dual-Model Extraction & Consensus Filtering
# -------------------------------------------------------------------
df_unlabeled = pd.read_csv("/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection/train.csv")
reports_list = df_unlabeled['Report'].fillna('').astype(str).tolist()
uids_list = df_unlabeled['StudyInstanceUID'].tolist()

df_gemma = run_model_inference("google/gemma-4-E2B-it", reports_list, uids_list)
df_qwen = run_model_inference("microsoft/Phi-4-mini-instruct", reports_list, uids_list)


# 3. Compute Strict Consensus
consensus_rows = []
total_entries = 0
agreed_entries = 0

for uid in uids_list:
    row = {'StudyInstanceUID': uid}
    for col in LABEL_COLS:
        q_val = df_qwen.loc[uid, col]
        g_val = df_gemma.loc[uid, col]

        total_entries += 1
        # Require identical values (both 0 or both 1)
        if q_val == g_val and q_val in [0, 1]:
            row[col] = float(q_val)
            agreed_entries += 1
        else:
            row[col] = np.nan # Disagreement, ambiguity, or null -> Safely Mask Out

    consensus_rows.append(row)

df_consensus = pd.DataFrame(consensus_rows)
df_consensus.to_csv("pseudo_labels_dual_consensus.csv", index=False)

print(f"\nConsensus Complete!")
print(f"Accepted Verified Labels: {agreed_entries} / {total_entries} ({agreed_entries/total_entries*100:.2f}%)")
print(f"Masked Out (NaN): {total_entries - agreed_entries} / {total_entries} ({(total_entries - agreed_entries)/total_entries*100:.2f}%)")


--- Loading Model: google/gemma-4-E2B-it ---


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Inferencing gemma-4-E2B-it:   0%|          | 0/276 [00:41<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

# 1. Configuration & Paths
IS_KAGGLE = False  # Set to True inside Kaggle environment
MODEL_PATH = "/kaggle/input/your-qwen-dataset/qwen2_5_1_5b_kaggle" if IS_KAGGLE else "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABEL_COLS = ['ACL', 'PCL', 'MCL', 'LCL', 'Medial_Meniscus', 'Lateral_Meniscus', 'Effusion']

# 2. Load Tokenizer with Left Padding (Required for Causal LM Batching)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Load Quantized Model (4-bit for speed & low VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

# 4. Prompt Template
SYSTEM_PROMPT = """You are a multilingual expert radiologist extracting findings from knee MRI reports in ANY language.

Assign values for each condition:
- 1: Explicitly positive / confirmed tear, injury, or effusion.
- 0: Explicitly normal, intact, absent, or negated (e.g., "sin rotura", "normal", "intacto", "unremarkable").
- null: Ambiguous, unclear, not mentioned, or uncertain.

CRITICAL: If uncertain or not explicitly stated, output null.

Respond strictly with a JSON object containing keys: ACL, PCL, MCL, LCL, Medial_Meniscus, Lateral_Meniscus, Effusion.
Example output:
{"ACL": 1, "PCL": 0, "MCL": null, "LCL": null, "Medial_Meniscus": 0, "Lateral_Meniscus": 1, "Effusion": 0}"""

def format_prompts(reports):
    batch_prompts = []
    for r in reports:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"MRI Report:\n\"\"\"{r}\"\"\""}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_prompts.append(prompt)
    return batch_prompts

def parse_json_response(text):
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return json.loads(text)
    except Exception:
        return {col: None for col in LABEL_COLS}

# 5. Batched Inference Execution Loop
def run_slm_extraction(df, batch_size=16):
    all_reports = df['Report'].fillna('').astype(str).tolist()
    uids = df['StudyInstanceUID'].tolist()
    parsed_results = []

    for i in tqdm(range(0, len(all_reports), batch_size), desc="Processing Batches"):
        batch_reports = all_reports[i:i+batch_size]
        batch_uids = uids[i:i+batch_size]

        prompts = format_prompts(batch_reports)
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=96,
                temperature=0.01,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # Decode generated text (trim prompt tokens)
        input_length = inputs['input_ids'].shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_length:], skip_special_tokens=True)

        for uid, response in zip(batch_uids, decoded):
            extracted = parse_json_response(response)
            row_dict = {'StudyInstanceUID': uid}
            for col in LABEL_COLS:
                val = extracted.get(col, None)
                row_dict[col] = float(val) if val in [0, 1] else np.nan
            parsed_results.append(row_dict)

    return pd.DataFrame(parsed_results)

In [ ]:
# Process all 4,349 unlabeled reports offline
df_train = pd.read_csv("/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection/train.csv")
unlabeled_df = df_train[df_train['ACL'].isnull()].reset_index(drop=True)

slm_labels_df = run_slm_extraction(unlabeled_df, batch_size=16)
slm_labels_df.to_csv("pseudo_labels_slm.csv", index=False)